In [1]:
import os
import sys
from pathlib import Path
import importlib
import torch

In [2]:
#@title Mount Goole Drive

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}


mount_drive = True  #@param {type:"boolean"}

clone_repo = False  #@param {type:"boolean"}

if clone_repo and mount_drive:

    from google.colab import drive
    drive.mount("/content/drive")

    root_path = os.path.join(root_path, "Flood-Mapping")

    !git clone https://github.com/TAX2310/Flood-Mapping.git $root_path

    sys.path.append(os.path.join(root_path))

    from S2.s2_config import S1_CFG
    cfg = S2_CFG()

    cfg.ROOT = Path(root_path)

elif not clone_repo and mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from src.S2.s2_config import S2_CFG
    cfg = S2_CFG()

    cfg.ROOT = Path(root_path)

elif clone_repo and not mount_drive:
    root_path = "Flood-Mapping"

    !git clone https://github.com/TAX2310/Flood-Mapping.git

    sys.path.append(root_path)

    from src.S2.s2_config import S2_CFG
    cfg = S2_CFG()

    cfg.ROOT = Path(root_path)

else:
    from src.S2.s2_config import S2_CFG
    cfg = S2_CFG()

    sys.path.append(os.path.join(cfg.ROOT))

cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Mounted at /content/drive


In [3]:
requirements = cfg.ROOT / "requirements.txt"
!pip install -r {requirements}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.9 MB/s eta 0:00:00


In [4]:
import src.S2.training.training as training
import src.util.io as io

In [5]:
cfg.DATASET = "STURM-fusion-24"

cfg.MODEL = "unet_resnet34_optical"

cfg.EPOCHS = 20

cfg.LR = 1e-3

cfg.BATCH_SIZE = 32

cfg.DROPOUT_RATE = 0.0

cfg.WEIGHT_DECAY = 0.0

cfg.USE_ROTATIONS = False

cfg.SPLIT_METHOD = 'by_event'

training.train_from_file(cfg)

Experiment directory: /content/drive/MyDrive/MSc/Flood-Mapping/experiments/STURM-fusion-24/Sentinel2_Optical/unet_resnet34_optical__by_event__rotation_False/lr_0.001__bs_32__e_20

Train: 100%|██████████| 44/44 [06:27<00:00,  3.55s/it, loss=0.2471]
                                                                   

Val: 100%|██████████| 9/9 [00:56<00:00,  4.16s/it, loss=0.3081]
                                                               
Epoch 1/20 - Train Loss: 0.3179 - Val Loss: 0.3152 - Val IoU: 0.6099 - Val F1: 0.7545

Train: 100%|██████████| 44/44 [03:28<00:00,  2.75s/it, loss=0.2494]
                                                                   

Val: 100%|██████████| 9/9 [00:49<00:00,  4.37s/it, loss=0.3429]
                                                               
Epoch 2/20 - Train Loss: 0.2245 - Val Loss: 0.3471 - Val IoU: 0.5880 - Val F1: 0.7343

Train: 100%|██████████| 44/44 [03:27<00:00,  2.35s/it, loss=0.2192]
                                                

<Popen: returncode: 0 args: ['/usr/bin/python3', '-u', '/content/drive/MyDri...>

In [6]:
from pathlib import Path
import numpy as np
import rasterio


def replace_nan_with_zero_in_dir(input_dir, output_dir=None, overwrite=False):
    """
    Replace NaN values with 0 in all .tif files in a directory.

    Parameters
    ----------
    input_dir : str or Path
        Directory containing .tif files.

    output_dir : str or Path, optional
        Directory to save cleaned files.
        If None and overwrite=False, creates a 'nan_fixed' folder inside input_dir.

    overwrite : bool
        If True, overwrites the original files.
        If False, saves cleaned copies to output_dir.
    """

    input_dir = Path(input_dir)

    if output_dir is None and not overwrite:
        output_dir = input_dir / "nan_fixed"

    if not overwrite:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    tif_files = list(input_dir.glob("*.tif"))

    print(f"Found {len(tif_files)} tif files")

    for tif_path in tif_files:
        with rasterio.open(tif_path) as src:
            image = src.read()
            profile = src.profile.copy()

        nan_count = np.isnan(image).sum()

        if nan_count == 0:
            print(f"No NaNs: {tif_path.name}")
            continue

        image = np.nan_to_num(image, nan=0.0)

        if overwrite:
            save_path = tif_path
        else:
            save_path = output_dir / tif_path.name

        with rasterio.open(save_path, "w", **profile) as dst:
            dst.write(image)

        print(f"Fixed {tif_path.name}: replaced {nan_count} NaNs")

    print("Done.")

In [7]:
#replace_nan_with_zero_in_dir(cfg.S2_PATH, output_dir=cfg.S2_PATH, overwrite=True)

In [ ]:
import time
import os

time.sleep(180)  # wait 3 minutes
os.kill(os.getpid(), 9)